In [1]:
import pandas as pd
import gc

def load_and_clean_data(file_path):
    """Загружает и очищает данные из Excel"""
    df = pd.read_excel(file_path)
    df.columns = [col.lower() for col in df.columns]
    df['produced_material_production_type'] = df['produced_material_production_type'].fillna('NULL')
    df['component_material_production_type'] = df['component_material_production_type'].fillna('NULL')
    print(f"Загружено строк: {len(df)}")
    print(f"Годы в данных: {sorted(df['year'].unique())}")
    return df

def get_unique_bom(df):
    """Получает уникальные связи BoM (без дубликатов по месяцам)"""
    unique_bom = df[['plant_id', 'year', 'produced_material',
                     'produced_material_release_type', 'produced_material_production_type',
                     'component_material', 'component_material_release_type',
                     'component_material_production_type']].drop_duplicates()
    print(f"Уникальных связей BoM: {len(unique_bom)}")
    return unique_bom

def build_bom_hierarchy(unique_bom):
    """Строит иерархию BoM через последовательные merge"""
    fin_df = unique_bom[unique_bom['produced_material_release_type'] == 'FIN'][
        ['plant_id', 'year', 'produced_material', 'produced_material_release_type',
         'produced_material_production_type']
    ].drop_duplicates().copy()
    fin_df.columns = ['plant_id', 'year', 'fin_material', 'fin_release_type', 'fin_production_type']
    print(f"FIN материалов: {len(fin_df)}")

    comp_df = unique_bom[['plant_id', 'year', 'produced_material', 'component_material',
                          'component_material_release_type', 'component_material_production_type']].copy()
    comp_df.columns = ['plant_id', 'year', 'parent_material', 'child_material',
                       'child_release_type', 'child_production_type']

    # 4 уровня merge (FIN→PROD1→PROD2→PROD3→PROD4)
    level1 = fin_df.merge(comp_df, left_on=['plant_id', 'year', 'fin_material'],
                          right_on=['plant_id', 'year', 'parent_material'], how='inner')
    level1 = level1.drop(columns=['parent_material']).rename(columns={
        'child_material': 'prod_1', 'child_release_type': 'prod_1_release', 'child_production_type': 'prod_1_type'})

    level2 = level1.merge(comp_df, left_on=['plant_id', 'year', 'prod_1'],
                          right_on=['plant_id', 'year', 'parent_material'], how='inner')
    level2 = level2.drop(columns=['parent_material']).rename(columns={
        'child_material': 'prod_2', 'child_release_type': 'prod_2_release', 'child_production_type': 'prod_2_type'})

    level3 = level2.merge(comp_df, left_on=['plant_id', 'year', 'prod_2'],
                          right_on=['plant_id', 'year', 'parent_material'], how='inner')
    level3 = level3.drop(columns=['parent_material']).rename(columns={
        'child_material': 'prod_3', 'child_release_type': 'prod_3_release', 'child_production_type': 'prod_3_type'})

    level4 = level3.merge(comp_df, left_on=['plant_id', 'year', 'prod_3'],
                          right_on=['plant_id', 'year', 'parent_material'], how='inner')
    level4 = level4.drop(columns=['parent_material']).rename(columns={
        'child_material': 'prod_4', 'child_release_type': 'prod_4_release', 'child_production_type': 'prod_4_type'})

    # Собираем все пары (PROD, Component)
    results = []
    for level, p_from, p_to in [(level2, 'prod_1', 'prod_2'), (level3, 'prod_2', 'prod_3'), (level4, 'prod_3', 'prod_4')]:
        temp = level[['plant_id', 'year', 'fin_material', 'fin_release_type', 'fin_production_type',
                      p_from, f'{p_from}_release', f'{p_from}_type',
                      p_to, f'{p_to}_release', f'{p_to}_type']].copy()
        temp.columns = ['plant', 'year', 'fin_id', 'fin_release', 'fin_type',
                        'prod_id', 'prod_release', 'prod_type',
                        'comp_id', 'comp_release', 'comp_type']
        results.append(temp)

    bom_structure = pd.concat(results, ignore_index=True)
    print(f"Структура BoM: {bom_structure.shape} строк")
    return bom_structure

def add_quantities(bom_structure, df):
    """Добавляет количества из исходных данных"""
    fin_qty_dict = {}
    comp_qty_dict = {}
    for _, row in df.iterrows():
        key = (row['plant_id'], row['year'], row['produced_material'])
        if key not in fin_qty_dict:
            fin_qty_dict[key] = row['produced_material_quantity']
        comp_key = (row['plant_id'], row['year'], row['produced_material'], row['component_material'])
        if comp_key not in comp_qty_dict:
            comp_qty_dict[comp_key] = row['component_material_quantity']

    bom_structure['fin_production_quantity'] = bom_structure.apply(
        lambda r: fin_qty_dict.get((r['plant'], r['year'], r['fin_id'])), axis=1)
    bom_structure['prod_material_production_quantity'] = bom_structure.apply(
        lambda r: fin_qty_dict.get((r['plant'], r['year'], r['prod_id'])), axis=1)
    bom_structure['component_consumption_quantity'] = bom_structure.apply(
        lambda r: comp_qty_dict.get((r['plant'], r['year'], r['prod_id'], r['comp_id'])), axis=1)
    return bom_structure

def format_final_output(bom_structure):
    """Форматирует финальный вывод"""
    bom_exploded = bom_structure.rename(columns={
        'fin_id': 'fin_material_id', 'fin_release': 'fin_material_release_type',
        'fin_type': 'fin_material_production_type',
        'prod_id': 'prod_material_id', 'prod_release': 'prod_material_release_type',
        'prod_type': 'prod_material_production_type',
        'comp_id': 'component_id', 'comp_release': 'component_material_release_type',
        'comp_type': 'component_material_production_type'
    })
    final_columns = [
        'plant', 'year',
        'fin_material_id', 'fin_material_release_type', 'fin_material_production_type',
        'fin_production_quantity',
        'prod_material_id', 'prod_material_release_type', 'prod_material_production_type',
        'prod_material_production_quantity',
        'component_id', 'component_material_release_type', 'component_material_production_type',
        'component_consumption_quantity'
    ]
    return bom_exploded[final_columns]

In [2]:
df = load_and_clean_data('../data/task_2_data_ex.xlsx')
unique_bom = get_unique_bom(df)
bom_structure = build_bom_hierarchy(unique_bom)
bom_structure = add_quantities(bom_structure, df)
bom_exploded = format_final_output(bom_structure)

print(f"\n=== ИТОГО ===")
print(f"Размер таблицы: {bom_exploded.shape}")
display(bom_exploded[bom_exploded['fin_material_id'] == 10000].head(15))

bom_exploded.to_excel('bom_exploded_result.xlsx', index=False)
print(f"\nСохранено в data/bom_exploded_result.xlsx")

Загружено строк: 1320
Годы в данных: [np.int64(2024)]
Уникальных связей BoM: 110
FIN материалов: 10
Структура BoM: (80, 11) строк

=== ИТОГО ===
Размер таблицы: (80, 14)


,plant,year,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_material_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity
0,RLT_10,2024,10000,FIN,8002,990.0,50000,PROD,8002.0,859.0,80070,PROD,8007.0,879.0
1,RLT_10,2024,10000,FIN,8002,990.0,50000,PROD,8002.0,859.0,90000,ADD,NULL,50.0
2,RLT_10,2024,10000,FIN,8002,990.0,50000,PROD,8002.0,859.0,90001,ADD,NULL,20.0
30,RLT_10,2024,10000,FIN,8002,990.0,80070,PROD,8007.0,929.0,80010,PROD,8001.0,3626.0
33,RLT_10,2024,10000,FIN,8002,990.0,80070,PROD,8007.0,929.0,90002,ADD,NULL,30.0
36,RLT_10,2024,10000,FIN,8002,990.0,80070,PROD,8007.0,929.0,90003,ADD,NULL,11.0
60,RLT_10,2024,10000,FIN,8002,990.0,80010,PROD,8001.0,1726.0,80000,PROD,8000.0,1818.0
61,RLT_10,2024,10000,FIN,8002,990.0,80010,PROD,8001.0,1726.0,90004,ADD,NULL,101.0



Сохранено в data/bom_exploded_result.xlsx
